In [ ]:
import requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import matplotlib.pyplot as plt
import ee
import geemap
import math
from google.colab import userdata

In [ ]:
#  Base URL for the CoreStack  API end point
base_url = 'https://geoserver.core-stack.org/api/v1/'

# Get generated layers
get_active_locations_endpoint = 'get_active_locations/'

get_generated_layers_endpoint = "get_generated_layer_urls"

# Add your API key here.
# You can generate an API key by logging into https://dashboard.core-stack.org with an Organization Admin account.
# Only Org Admins have permission to generate API keys.
# If you are not an Org Admin, please contact your admin to obtain access.
# Api_key = userdata.get('API_key')

# Define the HTTP request headers with the API key
headers = {
    "X-API-Key": ""
}

In [ ]:
import requests
import json

generated_layer_api_url = f"{base_url}{get_active_locations_endpoint}"

def get_active_locations(api_url, headers):
    """
    Fetch and format active locations from the API
    """
    try:
        response = requests.get(api_url, headers=headers)
        response.raise_for_status()
        data = response.json()

        formatted_data = []

        for state in data:
            state_info = {
                "state": state['label'],
                "state_id": state['state_id'],
                "districts": []
            }

            for district in state.get('district', []):
                district_info = {
                    "district": district['label'],
                    "district_id": district['district_id'],
                    "tehsils": []
                }

                for block in district.get('blocks', []):
                    district_info["tehsils"].append({
                        "tehsil": block['label'],
                        "tehsil_id": block['tehsil_id'],
                        "block_id": block['block_id']
                    })

                state_info["districts"].append(district_info)

            formatted_data.append(state_info)

        return formatted_data

    except Exception as e:
        return {"error": str(e)}

# Usage
locations = get_active_locations(generated_layer_api_url, headers)

# Print formatted JSON
print(json.dumps(locations, indent=2))

# Or save to file
with open('active_locations.json', 'w') as f:
    json.dump(locations, f, indent=2)

In [ ]:
##
state = 'Bihar'
district = 'Jamui'
tehsil = 'Jamui'

In [ ]:
import re
def valid_gee_text(description):
    description = re.sub(r"[^a-zA-Z0-9 .,:;_-]", "", description)
    return description.replace(" ", "_")


# Function to create arrow geometry
import math

def create_arrow(feature):
    """Create custom arrow geometry from MWS centroid"""

    direction = ee.String(feature.get('direction'))

    # FORCE centroid (polygon-safe)
    geom = feature.geometry().centroid()
    coords = geom.coordinates()
    lon = ee.Number(coords.get(0))
    lat = ee.Number(coords.get(1))

    # Arrow parameters (degree-based, visible)
    length = ee.Number(0.02)
    arrowhead_length = ee.Number(0.003)
    arrowhead_angle = ee.Number(35)

    def make_arrow(dir_num, angle_deg):
        angle_rad = ee.Number(angle_deg).multiply(math.pi).divide(180)

        dx = length.multiply(angle_rad.cos())
        dy = length.multiply(angle_rad.sin())

        end_lon = lon.add(dx)
        end_lat = lat.add(dy)

        shaft = ee.Geometry.LineString([[lon, lat], [end_lon, end_lat]])

        left_angle = ee.Number(angle_deg).add(180).subtract(arrowhead_angle)
        left_rad = left_angle.multiply(math.pi).divide(180)
        left_dx = arrowhead_length.multiply(left_rad.cos())
        left_dy = arrowhead_length.multiply(left_rad.sin())

        left_wing = ee.Geometry.LineString([
            [end_lon, end_lat],
            [end_lon.add(left_dx), end_lat.add(left_dy)]
        ])

        right_angle = ee.Number(angle_deg).add(180).add(arrowhead_angle)
        right_rad = right_angle.multiply(math.pi).divide(180)
        right_dx = arrowhead_length.multiply(right_rad.cos())
        right_dy = arrowhead_length.multiply(right_rad.sin())

        right_wing = ee.Geometry.LineString([
            [end_lon, end_lat],
            [end_lon.add(right_dx), end_lat.add(right_dy)]
        ])

        return ee.FeatureCollection([
            ee.Feature(shaft, {'direction': dir_num}),
            ee.Feature(left_wing, {'direction': dir_num}),
            ee.Feature(right_wing, {'direction': dir_num}),
        ])

    return ee.Algorithms.If(
        direction.equals('1'), make_arrow('1', 45),
        ee.Algorithms.If(
            direction.equals('2'), make_arrow('2', 0),
            ee.Algorithms.If(
                direction.equals('3'), make_arrow('3', 315),
                ee.Algorithms.If(
                    direction.equals('4'), make_arrow('4', 270),
                    ee.Algorithms.If(
                        direction.equals('5'), make_arrow('5', 225),
                        ee.Algorithms.If(
                            direction.equals('6'), make_arrow('6', 180),
                            ee.Algorithms.If(
                                direction.equals('7'), make_arrow('7', 135),
                                ee.Algorithms.If(
                                    direction.equals('8'), make_arrow('8', 90),
                                    ee.FeatureCollection([])
                                )
                            )
                        )
                    )
                )
            )
        )
    )



In [ ]:
import io
import math
import requests
import geopandas as gpd
import folium
from collections import Counter


# ──────────────────────────────────────────────────────────────────────────────
# Direction metadata (kept for polygon fill colour + legend)
# ──────────────────────────────────────────────────────────────────────────────

DIRECTION_META = {
    "1": {"label": "NE", "color": "#e63946", "angle_deg": 45},
    "2": {"label": "N",  "color": "#2a3a8a", "angle_deg": 0},
    "3": {"label": "NW", "color": "#186a85", "angle_deg": 315},
    "4": {"label": "W",  "color": "#1a8a55", "angle_deg": 270},
    "5": {"label": "SW", "color": "#5a8a20", "angle_deg": 225},
    "6": {"label": "S",  "color": "#9a7a1a", "angle_deg": 180},
    "7": {"label": "SE", "color": "#aa4a10", "angle_deg": 135},
    "8": {"label": "E",  "color": "#8a1a00", "angle_deg": 90},
}

GEOSERVER_MWS_BASE       = "https://geoserver.core-stack.org:8443/geoserver/mws/ows"
GEOSERVER_CENTROID_BASE  = "https://geoserver.core-stack.org:8443/geoserver/mws_centroid/ows"


# ──────────────────────────────────────────────────────────────────────────────
# Arrow drawing  (start → end, real coordinates)
# ──────────────────────────────────────────────────────────────────────────────

# Fixed arrowhead size in degrees (~400m at Indian latitudes)
ARROW_HEAD_LENGTH = 0.004
ARROW_HEAD_WING   = 0.002


def draw_arrowhead(map_obj, tip, direction_lon, direction_lat, color):
    """
    Fixed-size arrowhead at `tip`, pointing in the direction of
    (direction_lon, direction_lat) → tip.
    Always the same visual size regardless of total arrow length.
    """
    dx = tip[0] - direction_lon
    dy = tip[1] - direction_lat
    length = math.sqrt(dx**2 + dy**2) or 1e-9
    ux, uy = dx / length, dy / length   # unit vector along arrow direction
    px, py = -uy, ux                    # perpendicular

    # Base centre = tip minus fixed head length along shaft direction
    base_lon = tip[0] - ux * ARROW_HEAD_LENGTH
    base_lat = tip[1] - uy * ARROW_HEAD_LENGTH

    left  = (base_lon + px * ARROW_HEAD_WING, base_lat + py * ARROW_HEAD_WING)
    right = (base_lon - px * ARROW_HEAD_WING, base_lat - py * ARROW_HEAD_WING)

    folium.Polygon(
        locations=[
            (tip[1],   tip[0]),
            (left[1],  left[0]),
            (right[1], right[0]),
        ],
        color=color, fill=True, fill_color=color, fill_opacity=1.0, weight=0,
    ).add_to(map_obj)


def add_flow_arrow(map_obj, start_lon, start_lat, end_lon, end_lat, color):
    """
    Shaft runs from start to (end - fixed head length).
    Arrowhead is always the same fixed size at the tip.
    """
    dx = end_lon - start_lon
    dy = end_lat - start_lat
    total_length = math.sqrt(dx**2 + dy**2) or 1e-9
    ux, uy = dx / total_length, dy / total_length

    # Shaft end = tip minus the fixed arrowhead length
    shaft_end_lon = end_lon - ux * ARROW_HEAD_LENGTH
    shaft_end_lat = end_lat - uy * ARROW_HEAD_LENGTH

    # Shaft
    folium.PolyLine(
        locations=[(start_lat, start_lon), (shaft_end_lat, shaft_end_lon)],
        color=color, weight=2.5, opacity=1.0,
    ).add_to(map_obj)

    # Fixed-size arrowhead at the tip
    draw_arrowhead(map_obj, (end_lon, end_lat), shaft_end_lon, shaft_end_lat, color)


# ──────────────────────────────────────────────────────────────────────────────
# GeoServer fetch helpers
# ──────────────────────────────────────────────────────────────────────────────

def fetch_gdf(url, params=None, timeout=60):
    resp = requests.get(url, params=params, timeout=timeout)
    resp.raise_for_status()
    return gpd.read_file(io.BytesIO(resp.content))


# ──────────────────────────────────────────────────────────────────────────────
# Main map function
# ──────────────────────────────────────────────────────────────────────────────

def mws_connectivity_map(connectivity_geo_url, district, tehsil):
    """
    Parameters
    ----------
    connectivity_geo_url : str
        GeoServer URL for the mws_connectivity layer.
        Each feature has: uid, direction, downstream, upstream.
    district : str
        e.g. 'anugul'
    tehsil : str
        e.g. 'anugul'
    """
    district = district.lower()
    tehsil   = tehsil.lower()

    # ── STEP 1: Fetch polygon boundaries ───────────────────────────────────
    print("🌐 Fetching polygon boundaries …")
    gdf_boundary = fetch_gdf(GEOSERVER_MWS_BASE, params={
        "service":      "WFS",
        "version":      "1.0.0",
        "request":      "GetFeature",
        "typeName":     f"mws:mws_{district}_{tehsil}",
        "outputFormat": "application/json",
    })
    gdf_boundary["uid"] = gdf_boundary["uid"].astype(str)

    # ── STEP 2: Fetch connectivity (uid, direction, downstream) ────────────
    print(f"Fetching connectivity layer …")
    gdf_conn = fetch_gdf(connectivity_geo_url)
    gdf_conn["uid"]        = gdf_conn["uid"].astype(str)
    gdf_conn["direction"]  = gdf_conn["direction"].astype(str)
    gdf_conn["downstream"] = gdf_conn["downstream"].astype(str)

    # ── STEP 3: Fetch mws_centroid layer (uid → lon, lat) ──────────────────
    centroid_layer = f"mws_centroid:{district}_{tehsil}_mws_centroid"
    print(f"Fetching centroid layer")
    gdf_centroid = fetch_gdf(GEOSERVER_CENTROID_BASE, params={
        "service":      "WFS",
        "version":      "1.0.0",
        "request":      "GetFeature",
        "typeName":     centroid_layer,
        "outputFormat": "application/json",
    })
    gdf_centroid["uid"] = gdf_centroid["uid"].astype(str)

    # uid → (lon, lat) lookup from centroid layer
    uid_to_lonlat = {
        row["uid"]: (row["centroid_lon"], row["centroid_lat"])
        for _, row in gdf_centroid.iterrows()
    }

    # ── STEP 4: Join direction onto boundary GDF ───────────────────────────
    uid_to_direction   = dict(zip(gdf_conn["uid"], gdf_conn["direction"]))
    uid_to_downstream  = dict(zip(gdf_conn["uid"], gdf_conn["downstream"]))

    gdf_boundary["direction"]  = gdf_boundary["uid"].map(uid_to_direction).fillna("0")
    gdf_boundary["downstream"] = gdf_boundary["uid"].map(uid_to_downstream).fillna("")

    matched = (gdf_boundary["direction"] != "0").sum()

    # ── STEP 5: Initialise map ─────────────────────────────────────────────
    center     = gdf_boundary.geometry.centroid
    map_center = [center.y.mean(), center.x.mean()]
    fmap = folium.Map(location=map_center, zoom_start=12, tiles="CartoDB positron")

    # ── STEP 6: BACKGROUND — MWS polygons coloured by direction ───────────
    def mws_style(feature):
        d     = str(feature["properties"].get("direction", "0"))
        meta  = DIRECTION_META.get(d)
        color = meta["color"] if meta else "#aaaaaa"
        return {
            "fillColor":   color,
            "fillOpacity": 0.25,
            "color":       "#333333",
            "weight":      0.8,
        }

    folium.GeoJson(
        gdf_boundary.__geo_interface__,
        name="MWS Boundary",
        style_function=mws_style,
        tooltip=folium.GeoJsonTooltip(
            fields=["uid", "direction", "downstream"],
            aliases=["UID", "Direction", "Downstream"],
            sticky=False,
        ),
    ).add_to(fmap)

    # ── STEP 7: FOREGROUND — real flow arrows (start uid → downstream uid) ─
    arrow_group = folium.FeatureGroup(name="Flow Arrows", show=True)

    arrow_count  = 0
    skipped_no_downstream = 0
    skipped_no_centroid   = 0

    for _, row in gdf_conn.iterrows():
        uid        = str(row["uid"])
        downstream = str(row["downstream"])
        direction  = str(row["direction"])

        if direction == "0" or direction not in DIRECTION_META:
            continue

        # Skip if no downstream target
        if not downstream or downstream in ("nan", "None", "0"):
            skipped_no_downstream += 1
            continue

        # Both start and end must have centroids
        if uid not in uid_to_lonlat or downstream not in uid_to_lonlat:
            skipped_no_centroid += 1
            continue

        start_lon, start_lat = uid_to_lonlat[uid]
        end_lon,   end_lat   = uid_to_lonlat[downstream]

        color = DIRECTION_META[direction]["color"]
        add_flow_arrow(arrow_group, start_lon, start_lat, end_lon, end_lat, color)
        arrow_count += 1

    arrow_group.add_to(fmap)

    # ── STEP 8: Legend ─────────────────────────────────────────────────────
    legend_rows = "".join(
        f'<div style="display:flex;align-items:center;margin-bottom:4px;">'
        f'<span style="display:inline-block;width:14px;height:14px;background:{m["color"]};'
        f'margin-right:8px;border-radius:2px;flex-shrink:0;"></span>'
        f'<span>{m["label"]}</span></div>'
        for m in DIRECTION_META.values()
    )

    fmap.get_root().html.add_child(folium.Element(f"""
    <div style="position:fixed;bottom:40px;left:40px;z-index:1000;background:white;
        border:1px solid #ccc;border-radius:8px;padding:12px 16px;font-size:12px;
        font-family:sans-serif;box-shadow:2px 2px 8px rgba(0,0,0,0.2);min-width:110px;">
      <div style="font-weight:bold;margin-bottom:8px;">Flow Direction</div>
      {legend_rows}
    </div>
    """))

    folium.LayerControl(collapsed=False).add_to(fmap)
    print("MAP CREATED SUCCESSFULLY")
    return fmap

In [ ]:
# # Get MWS Connectivity
state_name = valid_gee_text(state.lower())
district_name = valid_gee_text(district.lower())
tehsil_name = valid_gee_text(tehsil.lower())
layer_name = f"{district}_{tehsil}_mws_connectivity"



params = {
  "state": state_name,
  "district": district_name,
  "tehsil": tehsil_name
}
generated_layer_api_url = f"{base_url}{get_generated_layers_endpoint}"

# Make the GET request with headers and parameters
response = requests.get(generated_layer_api_url, params=params, headers=headers)
mws_connectivity_geo_url = ''

for layer in response.json():
    if layer["layer_name"] == f"{district_name}_{tehsil_name}_mws_connectivity":
        mws_connectivity_geo_url = layer["layer_url"]
        break

print("MWS connectivity url", mws_connectivity_geo_url)

# Check if the request was successful (status code 200)
if response.status_code == 200:
    print("Inside response text")
    state_name = state
    district_name = district
    tehsil_name = tehsil
else:
    print(f"Layer not generated for the location: {response.status_code}")
    print("Response content:", response.text)


mws_connectivity_map(mws_connectivity_geo_url, district, tehsil)

MWS connectivity url https://geoserver.core-stack.org:8443/geoserver/mws_connectivity/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=mws_connectivity:jamui_jamui_mws_connectivity&outputFormat=application/json
Inside response text
🌐 Fetching polygon boundaries …
Fetching connectivity layer …
Fetching centroid layer


/tmp/ipython-input-3552779705.py:165: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center     = gdf_boundary.geometry.centroid


MAP CREATED SUCCESSFULLY
